In [ ]:
!pip install -q ultralytics roboflow
import torch
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:', torch.cuda.get_device_name(0))

In [ ]:
import shutil, random, yaml
from pathlib import Path
from google.colab import userdata
from roboflow import Roboflow

rf = Roboflow(api_key=userdata.get('ROBOFLOW_API_KEY'))
dataset = rf.workspace('car-license-plate-detection').project('license-plate-lprsn').version(18).download('yolov8')
yaml_path = dataset.location + '/data.yaml'
base = Path(dataset.location)

# create test split from 15% of train (before training)
random.seed(42)
(base / 'test' / 'images').mkdir(parents=True, exist_ok=True)
(base / 'test' / 'labels').mkdir(parents=True, exist_ok=True)
train_imgs = sorted((base / 'train' / 'images').glob('*.*'))
test_imgs = random.sample(train_imgs, len(train_imgs * 0.15))
for img in test_imgs:
    shutil.move(str(img), str(base / 'test' / 'images' / img.name))
    lbl = base / 'train' / 'labels' / (img.stem + '.txt')
    if lbl.exists():
        shutil.move(str(lbl), str(base / 'test' / 'labels' / lbl.name))

with open(yaml_path) as f:
    cfg = yaml.safe_load(f)
cfg['test'] = str(base / 'test' / 'images')
with open(yaml_path, 'w') as f:
    yaml.dump(cfg, f)

for split in ('train', 'valid', 'test'):
    n = len(list((base / split / 'images').glob('*.*')))
    print(f'{split:<8} {n} images')

In [ ]:
from ultralytics import YOLO

model = YOLO('yolov8n-seg.pt')
model.train(
    data=yaml_path,
    epochs=50,
    imgsz=416,
    batch=16,
    device=0,
    project='/content/runs',
    name='plate_segmentor',
    patience=10,
    exist_ok=True,
)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv('/content/runs/plate_segmentor/results.csv')
df.columns = df.columns.str.strip()
epochs = df['epoch']

fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for ax, (tr, vl, title) in zip(axes[:3], [
    ('train/box_loss', 'val/box_loss', 'Box Loss'),
    ('train/seg_loss', 'val/seg_loss', 'Seg Loss'),
    ('train/cls_loss', 'val/cls_loss', 'Cls Loss'),
]):
    ax.plot(epochs, df[tr], label='train')
    ax.plot(epochs, df[vl], label='val', linestyle='--')
    ax.set_title(title)
    ax.set_xlabel('Epoch')
    ax.legend()
    ax.grid(True, alpha=0.3)

map_col = 'metrics/mAP50(M)' if 'metrics/mAP50(M)' in df.columns else 'metrics/mAP50(B)'
axes[3].plot(epochs, df[map_col], color='green')
axes[3].set_title('mAP@0.5 (val)')
axes[3].set_xlabel('Epoch')
axes[3].set_ylim(0, 1)
axes[3].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/content/loss_curves.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
best_weights = '/content/runs/plate_segmentor/weights/best.pt'
trained = YOLO(best_weights)

m = trained.val(data=yaml_path, split='val')
print('=== Validation ===')
print(f'Box  mAP50:    {m.box.map50:.3f}')
print(f'Box  mAP50-95: {m.box.map:.3f}')
print(f'Mask mAP50:    {m.seg.map50:.3f}')
print(f'Mask mAP50-95: {m.seg.map:.3f}')

mt = trained.val(data=yaml_path, split='test')
print('=== Test ===')
print(f'Box  mAP50:    {mt.box.map50:.3f}')
print(f'Box  mAP50-95: {mt.box.map:.3f}')
print(f'Mask mAP50:    {mt.seg.map50:.3f}')
print(f'Mask mAP50-95: {mt.seg.map:.3f}')

In [ ]:
import shutil, os
from google.colab import drive

trained.export(format='onnx', imgsz=416, simplify=True, opset=12)
onnx_path = best_weights.replace('.pt', '.onnx')

drive.mount('/content/drive')
shutil.copy(onnx_path, '/content/drive/MyDrive/plate-segmentor.onnx')
shutil.copy('/content/loss_curves.png', '/content/drive/MyDrive/loss_curves.png')
print('Done')